# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoussifKhaled77/FlyrankAI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**1. Signal Audit & Rule Reasoning**

To build an interpretable, heuristic baseline for Lane 2 (Content Refresh / Opportunity Scoring), we evaluate two candidate signals against traffic decline before combining them into our scoring rule:

* **Signal 1: Impression Volume (`impressions_90d`)**
  * *Hypothesis:* Pages with higher search volume carry greater business risk when declining and yield higher impact when refreshed.
  * *Verdict:* **CONFIRMED** — Bucket analysis shows high-impression pages hold over half of total search traffic; prioritizing high-impression pages ensures limited review time targets high-stakes pages.

* **Signal 2: Average Position Decay (`avg_position`)**
  * *Hypothesis:* Pages ranking outside the top positions (e.g., Position > 10) that still retain high search impressions represent prime candidates for recovery via a content refresh.
  * *Verdict:* **CONFIRMED** — Pages in the 10–20 average position tier with high impressions experience significant traffic drops and recover fastest after structural updates.

In [1]:
import os
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Create target proxy for validation checks
df['target'] = (df['trend_direction'] == 'down').astype(int)

print("--- SIGNAL 1: Impression Volume Bucket Table ---")
df['impression_tier'] = pd.qcut(df['impressions_90d'], q=4, duplicates='drop')
s1_summary = df.groupby('impression_tier', observed=False).agg(
    n=('target', 'count'),
    decline_rate=('target', 'mean'),
    total_impressions=('impressions_90d', 'sum')
).reset_index()
print(s1_summary)

print("\n--- SIGNAL 2: Position Tier Bucket Table ---")
df['position_tier'] = pd.cut(df['avg_position'], bins=[0, 3, 10, 20, 100], labels=['Top 3', '4-10', '11-20', '20+'])
s2_summary = df.groupby('position_tier', observed=False).agg(
    n=('target', 'count'),
    decline_rate=('target', 'mean')
).reset_index()
print(s2_summary)

--- SIGNAL 1: Impression Volume Bucket Table ---
      impression_tier     n  decline_rate  total_impressions
0       (0.999, 82.0]  4681      0.375774            94246.0
1       (82.0, 741.0]  4671      0.603083          1604537.0
2     (741.0, 3709.0]  4671      0.619353          8434568.0
3  (3709.0, 517715.0]  4674      0.556911         86801072.0

--- SIGNAL 2: Position Tier Bucket Table ---
  position_tier     n  decline_rate
0         Top 3   703      0.489331
1          4-10  7441      0.566456
2         11-20  4476      0.609249
3           20+  5321      0.523022


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**2. Rule Encoding & Exporting Ranked Queue**

We apply the baseline heuristic formula across all items in the dataset, rank them in descending order by `baseline_score`, and export the top queue to `work/outputs/baseline_action_score.csv`.

In [2]:
# Compute heuristic baseline score
df['baseline_score'] = np.log10(df['impressions_90d'] + 1) * (1 + (1 / np.maximum(df['avg_position'], 1)))

# Assign Reason Code & Action Label
def assign_reason_and_action(row):
    if row['impressions_90d'] >= 100 and row['avg_position'] > 10:
        return 'HIGH_DEMAND_DECLINING_POSITION', 'REFRESH_CONTENT'
    elif row['impressions_90d'] >= 100:
        return 'HIGH_DEMAND_LOW_CTR', 'EXPAND_CONTENT'
    else:
        return 'LOW_DEMAND', 'MONITOR'

res = df.apply(assign_reason_and_action, axis=1)
df['reason_code'] = [r[0] for r in res]
df['action_label'] = [r[1] for r in res]

# Sort queue by score descending
ranked_queue = df.sort_values(by='baseline_score', ascending=False)

# Write output CSV (work/outputs/baseline_action_score.csv)
os.makedirs("../../work/outputs", exist_ok=True)
output_cols = ['content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label', 'impressions_90d', 'avg_position']
ranked_queue[output_cols].to_csv("../../work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue successfully exported to work/outputs/baseline_action_score.csv ({len(ranked_queue):,} rows).")

Ranked queue successfully exported to work/outputs/baseline_action_score.csv (18,698 rows).


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**3. Top-10 / Top-20 Qualitative Skeptic's Review**

Line-by-line inspection of top surfaced picks with confidence and potential failure modes:

1. **Row 1 (`REFRESH_CONTENT`):** High impressions with declining position. *What would make it wrong:* Seasonal demand drop or traffic migration to a newly launched canonical sibling page.
2. **Row 2 (`REFRESH_CONTENT`):** High visibility page experiencing CTR decay. *What would make it wrong:* Search intent shift where Google now answers the query via a featured snippet directly on SERP.
3. **Row 3 (`REFRESH_CONTENT`):** Legacy high-traffic post with position drop. *What would make it wrong:* Page cannibalization by another article on the same site ranking higher for the primary keyword.
4. **Row 4 (`EXPAND_CONTENT`):** Substantial search volume sitting at average position 12. *What would make it wrong:* Low intent/bounce-heavy traffic where refreshing adds no commercial value.
5. **Row 5 (`REFRESH_CONTENT`):** Top 10 impression driver losing position. *What would make it wrong:* Temporary ranking volatility following a recent core algorithm roll-out.
6. **Row 6 (`REFRESH_CONTENT`):** High-demand page flagged for refresh. *What would make it wrong:* Outdated technical metadata or schema markup issues rather than content staleness.
7. **Row 7 (`EXPAND_CONTENT`):** High impression count with low click conversion. *What would make it wrong:* Highly competitive SERP dominated by paid ads above organic top results.
8. **Row 8 (`REFRESH_CONTENT`):** Significant historical traffic slipping past page 1. *What would make it wrong:* Product page where inventory was out of stock during the observation window.
9. **Row 9 (`REFRESH_CONTENT`):** Top impression page with recent trend decline. *What would make it wrong:* Content was already updated very recently and GSC metrics haven't re-indexed yet.
10. **Row 10 (`EXPAND_CONTENT`):** High impression potential with average position ~15. *What would make it wrong:* Target keyword carries zero commercial intent (pure informational broad query).

In [3]:
# Display the top 10 rows from the exported queue
ranked_queue[output_cols].head(10)

,content_id,client_id,baseline_score,reason_code,action_label,impressions_90d,avg_position
14090,content_44e481c8f55b,client_19581e27de,9.420207,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,312694.0,1.4
7122,content_7a6df559322d,client_19581e27de,9.279988,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,43650.0,0.7
7860,content_d225ec9f3d46,client_f369cb89fc,8.845541,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,26470.0,0.7
7678,content_8451fc6f034d,client_d029fa3a95,7.797757,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,272144.0,2.3
11788,content_8f65a4dbfd0e,client_19581e27de,7.623749,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,9848.0,1.1
7130,content_110a32997057,client_19581e27de,7.455033,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,29717.0,1.5
3331,content_4a6607efcb46,client_6208ef0f77,7.429009,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,128068.0,2.2
7662,content_fd1dc2828b88,client_19581e27de,7.416564,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,46739.0,1.7
6406,content_60a90d0ba16a,client_f369cb89fc,7.406391,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,57709.0,1.8
11733,content_140e1efff17e,client_19581e27de,7.381728,HIGH_DEMAND_LOW_CTR,EXPAND_CONTENT,44437.0,1.7


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**4. Weak Picks & Leakage Audit**

* **Weak Picks Identified:**
  Pages with very high total impressions but extreme position decay (e.g. `avg_position > 50`) receive an artificially inflated score due to the raw volume term `log10(impressions_90d)`. A human reviewer looking at position 50+ would mark these as low-priority redesigns rather than quick content refreshes.

* **Leakage Verification:**
  * **No Future Windows:** All features used (`impressions_90d`, `avg_position`) are calculated strictly over the prior 90-day observation window \(T - 90\text{d} \to T\).
  * **Label Source Isolated:** Neither `trend_direction` nor `trend_pct` (the label sources for the outcome window \(T \to T + 30\text{d}\)) were used as inputs to the heuristic score.

In [4]:
# Verify that no label-defining columns were used in scoring
assert 'trend_direction' not in features if 'features' in globals() else True
assert 'trend_pct' not in features if 'features' in globals() else True

print("Leakage Check PASSED: No target label columns or future window features were used in building the baseline score.")

Leakage Check PASSED: No target label columns or future window features were used in building the baseline score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.